Imports

In [11]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors
import pickle

Load Data

In [12]:
df = pd.read_csv( 'recipes_dataset.csv', encoding='utf-8-sig')
df.shape

(80, 19)

 EDA

In [13]:
print(df.dtypes)
print(df.isnull().sum())
print(df['cuisine'].value_counts())

recipe_id             str
name_th               str
name_en               str
cuisine               str
origin                str
course                str
main_ingredients      str
cooking_method        str
prep_time_min       int64
cook_time_min       int64
total_time_min      int64
servings            int64
difficulty            str
spice_level         int64
calories_kcal       int64
allergens             str
flavor_profile        str
description           str
steps                 str
dtype: object
recipe_id           0
name_th             0
name_en             0
cuisine             0
origin              0
course              0
main_ingredients    0
cooking_method      0
prep_time_min       0
cook_time_min       0
total_time_min      0
servings            0
difficulty          0
spice_level         0
calories_kcal       0
allergens           0
flavor_profile      0
description         0
steps               0
dtype: int64
cuisine
Western    43
Thai       37
Name: count, dtype: int64

Ingredient Keywords

In [14]:
ingredient_keywords = {
    'has_pork':        ['หมูสับ', 'หมู'],
    'has_chicken':     ['อกไก่', 'ไก่'],
    'has_shrimp':      ['กุ้ง'],
    'has_bacon':       ['เบคอน', 'แฮม'],
    'has_egg':         ['ไข่'],
    'has_rice':        ['ข้าว'],
    'has_pasta':       ['พาสต้า', 'สปาเก็ต', 'เพนเน่', 'ลาซานญ่า'],
    'has_bread':       ['ขนมปัง'],
    'has_holy_basil':  ['กะเพรา'],
    'has_tomato':      ['มะเขือเทศ'],
    'has_cheese':      ['ชีส'],
    'has_cabbage':     ['ผักกาด'],
    'has_fish':        ['ปลากะพง', 'ปลาค็อด', 'ปลาแซลมอน', 'ปลาหมึก', 'ปลา'],
    'has_beef':        ['เนื้อวัว', 'เนื้อริบอาย', 'เนื้อสันคอ', 'เนื้อแกะ'],
    'has_shellfish':   ['ปูม้า', 'เนื้อปู', 'หอยลาย', 'หอยแมลงภู่'],
    'has_noodle':      ['เส้นจันท์', 'เส้นใหญ่', 'วุ้นเส้น', 'บะหมี่'],
    'has_coconut_milk':['กะทิ', 'น้ำกะทิ'],
    'has_mushroom':    ['เห็ด'],
}

Feature Engineering

In [15]:
for col_name, keywords in ingredient_keywords.items():
    df[col_name] = df['main_ingredients'].apply(
        lambda x: 1 if any(
            kw in item
            for item in x.split(';')
            for kw in keywords
            if not (item.strip().startswith('ไข่') and kw == 'ไก่')
        ) else 0
    )
print(df[list(ingredient_keywords.keys())].sum())

has_pork            12
has_chicken         11
has_shrimp           6
has_bacon            5
has_egg             18
has_rice            14
has_pasta            5
has_bread            4
has_holy_basil       1
has_tomato          10
has_cheese          15
has_cabbage          4
has_fish             8
has_beef             5
has_shellfish        4
has_noodle           6
has_coconut_milk     8
has_mushroom         9
dtype: int64


KNN Model

In [16]:
X = df[list(ingredient_keywords.keys())].values
knn_model = NearestNeighbors(n_neighbors=5, metric='cosine')
knn_model.fit(X)

,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
Name,Type,Value
effective_metric_ effective_metric_: strMetric used to compute distances to neighbors.,str,'cosine'
effective_metric_params_ effective_metric_params_: dictParameters for the metric used to compute distances to neighbors.,dict,{}


Recommend & Match Functions

In [17]:
def recommend_knn(user_ingredients, top_n=5):
    user_vector = [
        1 if any(kw in ing for ing in user_ingredients for kw in keywords)
        else 0
        for keywords in ingredient_keywords.values()
    ]
    distances, indices = knn_model.kneighbors([user_vector], n_neighbors=top_n)
    result = df[['name_th', 'name_en', 'cuisine']].iloc[indices[0]].copy()
    result['distance'] = distances[0].round(3)
    return result

def calculate_match(user_ingredients, recipe_index):
    recipe_vector = df[list(ingredient_keywords.keys())].iloc[recipe_index]
    recipe_needs = [col for col, val in recipe_vector.items() if val == 1]
    col_to_name = {
        'has_pork': 'หมู', 'has_chicken': 'ไก่', 'has_shrimp': 'กุ้ง',
        'has_bacon': 'เบคอน', 'has_egg': 'ไข่', 'has_rice': 'ข้าว',
        'has_pasta': 'พาสต้า', 'has_bread': 'ขนมปัง', 'has_holy_basil': 'กะเพรา',
        'has_tomato': 'มะเขือเทศ', 'has_cheese': 'ชีส', 'has_cabbage': 'ผักกาด',
        'has_fish': 'ปลา', 'has_beef': 'เนื้อ', 'has_shellfish': 'อาหารทะเล',
        'has_noodle': 'เส้น', 'has_coconut_milk': 'กะทิ', 'has_mushroom': 'เห็ด',
    }
    user_has = [col for col, keywords in ingredient_keywords.items()
                if any(kw in ing for ing in user_ingredients for kw in keywords)]
    have = [col_to_name[c] for c in recipe_needs if c in user_has]
    missing = [col_to_name[c] for c in recipe_needs if c not in user_has]
    match_pct = len(have) / len(recipe_needs) * 100 if recipe_needs else 0
    return round(match_pct, 1), have, missing

Test Functions

In [18]:
print(recommend_knn(['หมูสับ', 'ไข่', 'กะเพรา']))
pct, have, missing = calculate_match(['หมูสับ', 'ไข่', 'กะเพรา'], 0)
print(f"Match: {pct}%  มีแล้ว: {have}  ขาด: {missing}")

                  name_th                                            name_en  \
0   ผัดกะเพราหมูสับไข่ดาว  Pad Kra Pao Moo Sap (Stir-fried Holy Basil wit...   
10         ไข่เจียวหมูสับ  Khai Jiao Moo Sap (Thai Crispy Omelette with M...   
15        วีนเนอร์ชนิตเซล                  Wiener Schnitzel (Breaded Cutlet)   
24           ผัดซีอิ๊วหมู     Pad See Ew (Stir-fried Wide Noodles with Pork)   
8   หมูทอดกระเทียมพริกไทย         Moo Tod Kratiam (Garlic Pepper Fried Pork)   

    cuisine  distance  
0      Thai     0.000  
10     Thai     0.333  
15  Western     0.333  
24     Thai     0.333  
8      Thai     0.423  


Match: 100.0%  มีแล้ว: ['หมู', 'ไข่', 'กะเพรา']  ขาด: []


Evaluation

In [19]:
import time

test_cases = [
    {'ingredients': ['หมูสับ', 'ไข่', 'กะเพรา'],                              'expected_id': 'R001'},
    {'ingredients': ['เบคอน', 'ไข่', 'พาสต้า', 'ชีส'],                        'expected_id': 'R018'},
    {'ingredients': ['กุ้ง', 'ไข่'],                                           'expected_id': 'R017'},
    {'ingredients': ['อกไก่', 'ข้าว'],                                         'expected_id': 'R021'},
    {'ingredients': ['พาสต้า', 'มะเขือเทศ'],                                   'expected_id': 'R020'},
    {'ingredients': ['ขนมปัง', 'ชีส'],                                         'expected_id': 'R028'},
    {'ingredients': ['หมูสับ', 'ไข่', 'ผักกาด'],                              'expected_id': 'R035'},
    {'ingredients': ['อกไก่', 'เบคอน', 'ไข่', 'ชีส', 'มะเขือเทศ', 'ผักกาด'], 'expected_id': 'R040'},
    {'ingredients': ['กุ้ง', 'ข้าว'],                                          'expected_id': 'R026'},
    {'ingredients': ['พาสต้า', 'มะเขือเทศ', 'ชีส'],                           'expected_id': 'R024'},
]

results = []
for tc in test_cases:
    ings, exp = tc['ingredients'], tc['expected_id']
    t0 = time.time()
    r_knn = recommend_knn(ings, top_n=3)
    ms_knn = round((time.time() - t0) * 1000, 2)
    top1 = df.loc[r_knn.index[0], 'recipe_id']
    top3 = df.loc[r_knn.index, 'recipe_id'].tolist()
    results.append({'test': ', '.join(ings[:3]) + ('...' if len(ings) > 3 else ''),
                    'expected': exp, 'knn_top1': top1,
                    'hit1': top1 == exp, 'hit3': exp in top3, 'ms': ms_knn})

df_res = pd.DataFrame(results)
print(df_res[['test', 'expected', 'knn_top1', 'hit1', 'hit3']].to_string())
print(f"\nTop-1: {df_res['hit1'].mean()*100:.0f}%  Top-3: {df_res['hit3'].mean()*100:.0f}%  Avg: {df_res['ms'].mean():.2f}ms")

                     test expected knn_top1   hit1  hit3
0     หมูสับ, ไข่, กะเพรา     R001     R001   True  True
1   เบคอน, ไข่, พาสต้า...     R018     R018   True  True
2               กุ้ง, ไข่     R017     R017   True  True
3             อกไก่, ข้าว     R021     R021   True  True
4       พาสต้า, มะเขือเทศ     R020     R020   True  True
5             ขนมปัง, ชีส     R028     R028   True  True
6     หมูสับ, ไข่, ผักกาด     R035     R035   True  True
7    อกไก่, เบคอน, ไข่...     R040     R040   True  True
8              กุ้ง, ข้าว     R026     R026   True  True
9  พาสต้า, มะเขือเทศ, ชีส     R024     R032  False  True

Top-1: 90%  Top-3: 100%  Avg: 4.10ms


Save Model

In [ ]:
import os

model_bundle = {
    'knn_model':           knn_model,
    'df':                  df,
    'ingredient_keywords': ingredient_keywords,
}

with open('model.pkl', 'wb') as f:
    pickle.dump(model_bundle, f)

with open('model.pkl', 'rb') as f:
    loaded = pickle.load(f)
print("keys:", list(loaded.keys()))
print("features:", len(loaded['ingredient_keywords']))

keys: ['knn_model', 'df', 'ingredient_keywords']
features: 18
